<a href="https://colab.research.google.com/github/PMoshel/data-on-conflicts-and-spatial-characteristics-of-the-territory/blob/main/comparison_of_territories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

загрузка и проверка данных

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
from google.colab import files

# Если файл еще не загружен, раскомментируйте следующую строку:
# uploaded = files.upload()

# Загрузка данных
df = pd.read_excel('all_par.xlsx')

# Приведение названий столбцов и изменение порядка
new_column_order = ['ID', 'триггер', 'сущ_назначение', 'план_назначение',
                    'функции_окруж', 'год_мед', 'застройка_коэф',
                    'центральность', 'население', 'истор_здания', 'год_пост']

# Переименовываем и переупорядочиваем столбцы
df = df[new_column_order]

# Функция для очистки пробелов в данных
def clean_data(df):
    """ Очищает данные от лишних пробелов в текстовых столбцах и приводит к нижнему регистру """
    df_clean = df.copy()

    # Список текстовых столбцов для очистки
    text_columns = ['сущ_назначение', 'план_назначение', 'функции_окруж']

    for col in text_columns:
        if col in df_clean.columns:
            # Приводим к нижнему регистру
            df_clean[col] = df_clean[col].astype(str).str.lower()
            # Удаляем пробелы в начале и конце строк
            df_clean[col] = df_clean[col].str.strip()
            # Заменяем множественные пробелы на один
            df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)

    # Для числовых столбцов преобразуем строки с пробелами в числа
    numeric_columns = ['год_мед', 'застройка_коэф', 'центральность',
                       'население', 'истор_здания', 'год_пост']

    for col in numeric_columns:
        if col in df_clean.columns:
            # Если значения строковые (с пробелами), преобразуем
            if df_clean[col].dtype == 'object':
                # Удаляем пробелы и преобразуем в числа
                df_clean[col] = pd.to_numeric(df_clean[col].astype(str).str.replace(' ', ''), errors='coerce')

    # Преобразуем 'население' в целое число, если оно числовое
    if 'население' in df_clean.columns:
        if pd.api.types.is_numeric_dtype(df_clean['население']):
            df_clean['население'] = df_clean['население'].astype('Int64')  # Используем Int64 для поддержки NaN

    return df_clean

# Очищаем данные
df = clean_data(df)

# 1. Первые 5 строк данных
print("Первые 5 строк данных:")
print(df.head())
print("\n" + "="*80)

# 2. Компактная информация о данных
print("\nОБЩАЯ ИНФОРМАЦИЯ О ДАННЫХ:")
print(f"Количество строк: {df.shape[0]}")
print(f"Количество столбцов: {df.shape[1]}")
print("\nТИПЫ ДАННЫХ И ПРОПУСКИ:")
info_df = pd.DataFrame({
    'Тип данных': df.dtypes,
    'Не пропущено': df.count(),
    'Пропущено': df.isnull().sum(),
    '% пропусков': (df.isnull().sum() / len(df) * 100).round(1)
})
print(info_df)
print("\n" + "="*80)

# Сохраняем копию данных
data = df.copy()

# Список столбцов для анализа (все кроме ID и триггера)
analysis_columns = [col for col in data.columns if col not in ['ID', 'триггер']]
print("\nВведите количественно-качественные характеристики проектируемой территории.")
print(f"\nСтолбцы для анализа ({len(analysis_columns)}): {', '.join(analysis_columns)}")


Первые 5 строк данных:
   ID                                            триггер  \
0   1  Подготовка строительной площадки в лесу на Сир...   
1   2  Проект ЖК «Сосновый бор» компании «СД Альфа Ка...   
2   3  Обновленный Генплан Новосибирска от 24 марта, ...   
3   4  Снос нескольких зданий соцгорода и риск продол...   
4   5            Проект строительства новых объектов НГУ   

                                      сущ_назначение план_назначение  \
0  экологически-природная c активно растущими дер...           жилая   
1                                      рекреационная           жилая   
2                                      рекреационная           жилая   
3                                              жилая           жилая   
4  экологически-природная c активно растущими дер...      социальная   

                                       функции_окруж  год_мед  застройка_коэф  \
0  экологически-природная c активно растущими дер...   2013.0        0.029545   
1  экологически-приро

существующее функциональное использование территории

In [ ]:
# Код для сравнения по столбцу "сущ_назначение" вес

# 1. Выводим список функций
def print_categories():
    categories = [
        "Акваротия (береговые полосы, острова, пляжи, водоемы)",
        "Другая (строительство, сложно классифицировать)",
        "Другая негативная (кладбища, инженерные сооружения, свалки)",
        "Жилая (жилые дома, общежития, СНТ)",
        "Коммерческая (магазины, кафе, рестораны, рынки)",
        "Культурно-досуговая (театры, кино, библиотеки, музеи)",
        "Ландшафтно-рекреационная (парки, скверы, бульвары)",
        "Общественно-деловая (банки, офисы, административные здания)",
        "Промышленная (заводы, фабрики)",
        "Рекреационная (игровые площадки, спортивные объекты, места для отдыха)",
        "Сельскохозяйственная (поля, фермы)",
        "Социальная (школы, больницы, детские сады, университеты)",
        "Транспортная (парковки, гаражи, заправки)",
        "Экологически-природная (луга, пустыри, природные территории)",
        "Экологически-природная c активно растущими деревьями (леса, заросли)"
    ]

    for category in categories:
        print(category)

# Вызов функции
print("ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print("(можно ввести несколько функций через запятую)")
print_categories()
print("\n" + "="*80)

# 2. Пользователь вводит функцию для сравнения
print("\nВВЕДИТЕ СУЩЕСТВУЮЩИЕ ФУНКЦИИ ТЕРРИТОРИЙ:")
print("(можно ввести несколько функций через запятую)")
user_input = input(">>> ").strip()

# Разделяем введенные функции
user_functions = [f.strip().lower() for f in user_input.split(',') if f.strip()]
print(f"\nПоиск по функциям: {user_functions}")
print("\n"+"="*80)

# 3. Матрица схожести функций
similarity_matrix = {
    'акваротия': {
        'акваротия': 1.0,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.1,
        'коммерческая': 0.2,
        'культурно-досуговая': 0.3,
        'ландшафтно-рекреационная': 0.5,
        'общественно-деловая': 0.0,
        'промышленная': 0.0,
        'рекреационная': 0.4,
        'сельскохозяйственная': 0.3,
        'социальная': 0.1,
        'транспортная': 0.0,
        'экологически-природная': 0.4,
        'экологически-природная c активно растущими деревьями': 0.2
    },

    'другая': {
        'акваротия': 0.1,
        'другая': 1.0,
        'другая негативная': 0.3,
        'жилая': 0.1,
        'коммерческая': 0.2,
        'культурно-досуговая': 0.1,
        'ландшафтно-рекреационная': 0.1,
        'общественно-деловая': 0.2,
        'промышленная': 0.2,
        'рекреационная': 0.1,
        'сельскохозяйственная': 0.1,
        'социальная': 0.1,
        'транспортная': 0.2,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'другая негативная': {
        'акваротия': 0.0,
        'другая': 0.3,
        'другая негативная': 1.0,
        'жилая': 0.0,
        'коммерческая': 0.0,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.0,
        'общественно-деловая': 0.1,
        'промышленная': 0.5,
        'рекреационная': 0.0,
        'сельскохозяйственная': 0.0,
        'социальная': 0.0,
        'транспортная': 0.3,
        'экологически-природная': 0.0,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'жилая': {
        'акваротия': 0.1,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 1.0,
        'коммерческая': 0.5,
        'культурно-досуговая': 0.3,
        'ландшафтно-рекреационная': 0.3,
        'общественно-деловая': 0.4,
        'промышленная': 0.0,
        'рекреационная': 0.3,
        'сельскохозяйственная': 0.2,
        'социальная': 0.7,
        'транспортная': 0.2,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'коммерческая': {
        'акваротия': 0.2,
        'другая': 0.2,
        'другая негативная': 0.0,
        'жилая': 0.5,
        'коммерческая': 1.0,
        'культурно-досуговая': 0.7,
        'ландшафтно-рекреационная': 0.2,
        'общественно-деловая': 0.8,
        'промышленная': 0.1,
        'рекреационная': 0.4,
        'сельскохозяйственная': 0.1,
        'социальная': 0.3,
        'транспортная': 0.4,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'культурно-досуговая': {
        'акваротия': 0.3,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.3,
        'коммерческая': 0.7,
        'культурно-досуговая': 1.0,
        'ландшафтно-рекреационная': 0.4,
        'общественно-деловая': 0.6,
        'промышленная': 0.0,
        'рекреационная': 0.7,
        'сельскохозяйственная': 0.0,
        'социальная': 0.5,
        'транспортная': 0.0,
        'экологически-природная': 0.2,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'ландшафтно-рекреационная': {
        'акваротия': 0.5,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.3,
        'коммерческая': 0.2,
        'культурно-досуговая': 0.4,
        'ландшафтно-рекреационная': 1.0,
        'общественно-деловая': 0.2,
        'промышленная': 0.0,
        'рекреационная': 0.7,
        'сельскохозяйственная': 0.3,
        'социальная': 0.3,
        'транспортная': 0.0,
        'экологически-природная': 0.7,
        'экологически-природная c активно растущими деревьями': 0.7
    },

    'общественно-деловая': {
        'акваротия': 0.0,
        'другая': 0.2,
        'другая негативная': 0.1,
        'жилая': 0.4,
        'коммерческая': 0.8,
        'культурно-досуговая': 0.6,
        'ландшафтно-рекреационная': 0.2,
        'общественно-деловая': 1.0,
        'промышленная': 0.1,
        'рекреационная': 0.3,
        'сельскохозяйственная': 0.1,
        'социальная': 0.6,
        'транспортная': 0.3,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'промышленная': {
        'акваротия': 0.0,
        'другая': 0.2,
        'другая негативная': 0.5,
        'жилая': 0.0,
        'коммерческая': 0.1,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.0,
        'общественно-деловая': 0.1,
        'промышленная': 1.0,
        'рекреационная': 0.0,
        'сельскохозяйственная': 0.0,
        'социальная': 0.0,
        'транспортная': 0.7,
        'экологически-природная': 0.0,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'рекреационная': {
        'акваротия': 0.4,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.3,
        'коммерческая': 0.4,
        'культурно-досуговая': 0.7,
        'ландшафтно-рекреационная': 0.7,
        'общественно-деловая': 0.3,
        'промышленная': 0.0,
        'рекреационная': 1.0,
        'сельскохозяйственная': 0.1,
        'социальная': 0.5,
        'транспортная': 0.0,
        'экологически-природная': 0.5,
        'экологически-природная c активно растущими деревьями': 0.4
    },

    'сельскохозяйственная': {
        'акваротия': 0.3,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.2,
        'коммерческая': 0.1,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.3,
        'общественно-деловая': 0.1,
        'промышленная': 0.0,
        'рекреационная': 0.1,
        'сельскохозяйственная': 1.0,
        'социальная': 0.1,
        'транспортная': 0.0,
        'экологически-природная': 0.5,
        'экологически-природная c активно растущими деревьями': 0.5
    },

    'социальная': {
        'акваротия': 0.1,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.7,
        'коммерческая': 0.3,
        'культурно-досуговая': 0.5,
        'ландшафтно-рекреационная': 0.3,
        'общественно-деловая': 0.6,
        'промышленная': 0.0,
        'рекреационная': 0.5,
        'сельскохозяйственная': 0.1,
        'социальная': 1.0,
        'транспортная': 0.4,
        'экологически-природная': 0.2,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'транспортная': {
        'акваротия': 0.0,
        'другая': 0.2,
        'другая негативная': 0.3,
        'жилая': 0.2,
        'коммерческая': 0.4,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.0,
        'общественно-деловая': 0.3,
        'промышленная': 0.7,
        'рекреационная': 0.0,
        'сельскохозяйственная': 0.0,
        'социальная': 0.4,
        'транспортная': 1.0,
        'экологически-природная': 0.0,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'экологически-природная': {
        'акваротия': 0.4,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.1,
        'коммерческая': 0.1,
        'культурно-досуговая': 0.2,
        'ландшафтно-рекреационная': 0.7,
        'общественно-деловая': 0.1,
        'промышленная': 0.0,
        'рекреационная': 0.5,
        'сельскохозяйственная': 0.5,
        'социальная': 0.2,
        'транспортная': 0.0,
        'экологически-природная': 1.0,
        'экологически-природная c активно растущими деревьями': 0.9
    },

    'экологически-природная c активно растущими деревьями': {
        'акваротия': 0.2,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.1,
        'коммерческая': 0.0,
        'культурно-досуговая': 0.1,
        'ландшафтно-рекреационная': 0.7,
        'общественно-деловая': 0.0,
        'промышленная': 0.0,
        'рекреационная': 0.4,
        'сельскохозяйственная': 0.5,
        'социальная': 0.1,
        'транспортная': 0.0,
        'экологически-природная': 0.9,
        'экологически-природная c активно растущими деревьями': 1.0
    }
}

# 4. Функция для расчета схожести функций
def calculate_function_similarity(func1, func2):
    """Рассчитывает схожесть между двумя функциями"""
    # Если хотя бы одна функция отсутствует - возвращаем None
    if pd.isna(func1) or pd.isna(func2):
        return None

    # Точное совпадение
    if func1 == func2:
        return 1.0

    # Проверяем матрицу схожести
    if func1 in similarity_matrix and func2 in similarity_matrix[func1]:
        return similarity_matrix[func1][func2]
    elif func2 in similarity_matrix and func1 in similarity_matrix[func2]:
        return similarity_matrix[func2][func1]

    # Если нет в матрице - низкая схожесть
    return 0.1

# 5. Рассчитываем схожесть для каждой строки
results = []
for idx, row in data.iterrows():
    territory_func = row['сущ_назначение']

    # Если пользователь не ввел функции или функции отсутствуют
    if not user_functions or pd.isna(territory_func):
        similarity = None
    else:
        # Рассчитываем максимальную схожесть с введенными функциями
        max_similarity = None
        for user_func in user_functions:
            sim = calculate_function_similarity(territory_func, user_func)
            if sim is not None:
                if max_similarity is None or sim > max_similarity:
                    max_similarity = sim

        # Если нашлось хотя бы одно сравнение
        if max_similarity is not None:
            similarity = max_similarity * 100  # В процентах
        else:
            similarity = None

    results.append({
        'ID': row['ID'],
        'сходство_%': round(similarity, 1),
        'сущ_назначение': territory_func,
        'триггер': row['триггер']
    })


# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 6. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 МАКСИМАЛЬНО ПОХОЖИХ ТЕРРИТОРИЙ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 7. Сохраняем результаты для финального сравнения
# Создаем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['сущ_назначение'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")



ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:
(можно ввести несколько функций через запятую)
Акваротия (береговые полосы, острова, пляжи, водоемы)
Другая (строительство, сложно классифицировать)
Другая негативная (кладбища, инженерные сооружения, свалки)
Жилая (жилые дома, общежития, СНТ)
Коммерческая (магазины, кафе, рестораны, рынки)
Культурно-досуговая (театры, кино, библиотеки, музеи)
Ландшафтно-рекреационная (парки, скверы, бульвары)
Общественно-деловая (банки, офисы, административные здания)
Промышленная (заводы, фабрики)
Рекреационная (игровые площадки, спортивные объекты, места для отдыха)
Сельскохозяйственная (поля, фермы)
Социальная (школы, больницы, детские сады, университеты)
Транспортная (парковки, гаражи, заправки)
Экологически-природная (луга, пустыри, природные территории)
Экологически-природная c активно растущими деревьями (леса, заросли)


ВВЕДИТЕ СУЩЕСТВУЮЩИЕ ФУНКЦИИ ТЕРРИТОРИЙ:
(можно ввести несколько функций через запятую)
>>> Ландшафтно-рекреационная

Поиск по функция

проектируемое функциональное использование территории

In [ ]:
# Код для сравнения по столбцу "план_назначение" вес

# 1. Выводим список функций
print("ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print_categories()
print("\n" + "="*80)

# 2. Пользователь вводит проектируемые функции
print("\nВВЕДИТЕ ПРОЕКТИРУЕМЫЕ ФУНКЦИИ ТЕРРИТОРИИ:")
print("(можно ввести несколько функций через запятую)")
user_input = input(">>> ").strip()

# Разделяем введенные функции
user_functions = [f.strip().lower() for f in user_input.split(',') if f.strip()]
print(f"\nПоиск по функциям: {user_functions}")
print("\n"+"="*80)

# 3. Проверяем, существует ли матрица сходства, если нет - определяем
if 'similarity_matrix' not in globals():
    similarity_matrix = {
        # Та же матрица из предыдущего кода
    }

# 4. Используем ту же функцию для расчета схожести
def calculate_function_similarity(func1, func2):
    """Рассчитывает схожесть между двумя функциями"""
    # Если хотя бы одна функция отсутствует - возвращаем None
    if pd.isna(func1) or pd.isna(func2):
        return None

    # Точное совпадение
    if func1 == func2:
        return 1.0

    # Проверяем матрицу схожести
    if func1 in similarity_matrix and func2 in similarity_matrix[func1]:
        return similarity_matrix[func1][func2]
    elif func2 in similarity_matrix and func1 in similarity_matrix[func2]:
        return similarity_matrix[func2][func1]

    # Если нет в матрице - низкая схожесть
    return 0.1

# 5. Рассчитываем схожесть для каждой строки по столбцу "план_назначение"
results = []
for idx, row in data.iterrows():
    territory_func = row['план_назначение']

    # Если пользователь не ввел функции или функции отсутствуют
    if not user_functions or pd.isna(territory_func):
        similarity = None
    else:
        # Рассчитываем максимальную схожесть с введенными функциями
        max_similarity = None
        for user_func in user_functions:
            sim = calculate_function_similarity(territory_func, user_func)
            if sim is not None:
                if max_similarity is None or sim > max_similarity:
                    max_similarity = sim

        # Если нашлось хотя бы одно сравнение
        if max_similarity is not None:
            similarity = max_similarity * 100  # В процентах
        else:
            similarity = None

    results.append({
        'ID': row['ID'],
        'сходство_%': round(similarity, 1),
        'план_назначение': territory_func,
        'триггер': row['триггер']

    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 6. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 МАКСИМАЛЬНО ПОХОЖИХ ТЕРРИТОРИЙ ПО ПРОЕКТИРУЕМОМУ НАЗНАЧЕНИЮ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 7. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['план_назначение'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")

ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:
Акваротия (береговые полосы, острова, пляжи, водоемы)
Другая (строительство, сложно классифицировать)
Другая негативная (кладбища, инженерные сооружения, свалки)
Жилая (жилые дома, общежития, СНТ)
Коммерческая (магазины, кафе, рестораны, рынки)
Культурно-досуговая (театры, кино, библиотеки, музеи)
Ландшафтно-рекреационная (парки, скверы, бульвары)
Общественно-деловая (банки, офисы, административные здания)
Промышленная (заводы, фабрики)
Рекреационная (игровые площадки, спортивные объекты, места для отдыха)
Сельскохозяйственная (поля, фермы)
Социальная (школы, больницы, детские сады, университеты)
Транспортная (парковки, гаражи, заправки)
Экологически-природная (луга, пустыри, природные территории)
Экологически-природная c активно растущими деревьями (леса, заросли)


ВВЕДИТЕ ПРОЕКТИРУЕМЫЕ ФУНКЦИИ ТЕРРИТОРИИ:
(можно ввести несколько функций через запятую)
>>> .

Поиск по функциям: ['.']


ТОП-10 МАКСИМАЛЬНО ПОХОЖИХ ТЕРРИТОРИЙ ПО ПРОЕКТИРУЕМОМУ НА

функциональное использование прилегающих территорий

In [ ]:
# Код для сравнения по столбцу "функции_окруж" коэффициент Жаккара

# 1. Выводим список функций
print("ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print_categories()
print("\n" + "="*80)

# 2. Пользователь вводит функцию для сравнения
print("\nВВЕДИТЕ ФУНКЦИИ ПРИЛЕГАЮЩИХ ТЕРРИТОРИЙ В РАДИУСЕ 500 МЕТРОВ:")
print("(можно ввести несколько функций через запятую)")
user_input = input(">>> ").strip()

# Разделяем введенные функции
user_functions = [f.strip().lower() for f in user_input.split(',') if f.strip()]
print(f"\nПоиск по функциям: {user_functions}")
print("\n"+"="*80)

# 3. Функция для расчета коэффициента Жаккара
def jaccard_similarity(set1, set2):
    """Рассчитывает коэффициент Жаккара между двумя множествами"""
    # Если хотя бы одно множество отсутствует - возвращаем None
    if pd.isna(set1) or pd.isna(set2):
        return None

    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))

    if union == 0:
        return 0.0

    return intersection / union

# 4. Предобработка функций из таблицы
def preprocess_functions(func_string):
    """ Преобразует строку функций из таблицы в множество """
    if pd.isna(func_string):
        return None

    # Разделяем по запятой, очищаем от пробелов, приводим к нижнему регистру
    functions = [f.strip().lower() for f in str(func_string).split(',')]
    return set(functions)

# 5. Подготавливаем множество пользовательских функций
user_set = set(user_functions) if user_functions else None

# 6. Рассчитываем схожесть для каждой строки
results = []
for idx, row in data.iterrows():
    # Получаем функции из таблицы
    territory_funcs_raw = row['функции_окруж']
    territory_set = preprocess_functions(territory_funcs_raw)

    # Рассчитываем коэффициент Жаккара
    if user_set is None or territory_set is None:
        similarity = None
    else:
        similarity = jaccard_similarity(user_set, territory_set)

    # Преобразуем в проценты, если не None
    similarity_percent = round(similarity * 100, 1) if similarity is not None else None

    # Создаем строку для отображения функций территории
    if territory_set:
        territory_funcs_display = ', '.join(sorted(territory_set))
    else:
        territory_funcs_display = None

    results.append({
        'ID': row['ID'],
        'сходство_%': similarity_percent,
        'функции_окруж': territory_funcs_display,
        'триггер': row['триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 7. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ФУНКЦИЙ ОКРУЖЕНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 8. Сохраняем результаты для финального сравнения
if 'similarity_scores' not in globals():
    similarity_scores = {}

# Сохраняем коэффициенты сходства (делим на 100 для перевода в диапазон 0-1)
similarity_scores['функции_окруж'] = dict(zip(
    results_df['ID'],
    results_df['сходство_%'].apply(lambda x: x/100 if x is not None else None)
))

print("\n" + "="*80)
print("Результаты сохранены. Можете перейти к следующему критерию.")

ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:
Акваротия (береговые полосы, острова, пляжи, водоемы)
Другая (строительство, сложно классифицировать)
Другая негативная (кладбища, инженерные сооружения, свалки)
Жилая (жилые дома, общежития, СНТ)
Коммерческая (магазины, кафе, рестораны, рынки)
Культурно-досуговая (театры, кино, библиотеки, музеи)
Ландшафтно-рекреационная (парки, скверы, бульвары)
Общественно-деловая (банки, офисы, административные здания)
Промышленная (заводы, фабрики)
Рекреационная (игровые площадки, спортивные объекты, места для отдыха)
Сельскохозяйственная (поля, фермы)
Социальная (школы, больницы, детские сады, университеты)
Транспортная (парковки, гаражи, заправки)
Экологически-природная (луга, пустыри, природные территории)
Экологически-природная c активно растущими деревьями (леса, заросли)


ВВЕДИТЕ ФУНКЦИИ ПРИЛЕГАЮЩИХ ТЕРРИТОРИЙ В РАДИУСЕ 500 МЕТРОВ:
(можно ввести несколько функций через запятую)
>>> .

Поиск по функциям: ['.']


ТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВО

среднее значение годов застройки окружения

In [ ]:
# Код для сравнения по столбцу "год_мед" логарифмическая нормализованная разница

# 1. Пользователь вводит средний год постройки окружения
print("\nВВЕДИТЕ СРЕДНИЙ ГОД ЗАСТРОЙКИ В РАДИУСЕ 500 МЕТРОВ")
user_input = input(">>> ")

# Проверяем, что введено число
user_year = None
if user_input:
    try:
        user_year = int(user_input)
    except ValueError:
        print("Ошибка: введите целое число")

    print(f"\nСредний год постройки окружения: {user_year}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по годам
def calculate_year_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None

    # Преобразуем значения к целым числам
    user_val = int(user_val)
    table_val = int(table_val)
    min_val = int(min_val)
    max_val = int(max_val)

    # Проверяем, чтобы значения были положительными
    if user_val <= 0 or table_val <= 0 or min_val <= 0 or max_val <= 0:
        return None

    # Рассчитываем логарифмическую нормализованную разницу
    log_user = np.log(user_val)
    log_table = np.log(table_val)
    log_min = np.log(min_val)
    log_max = np.log(max_val)

    # Формула: 1 - (|log(PA) - log(PB)|) / (log(Pmax) - log(Pmin))
    similarity = 1 - (abs(log_user - log_table) / (log_max - log_min))

    return similarity

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение года в таблице
valid_years = data['год_мед'].dropna()
min_year = valid_years.min()
max_year = valid_years.max()

for idx, row in data.iterrows():
    # Получаем год из таблицы
    table_year = (row['год_мед'])


    # Рассчитываем схожесть
    if user_year is None or pd.isna(table_year):
        similarity = None
    else:
        similarity = calculate_year_similarity(user_year, table_year, min_year, max_year)

    # Преобразуем в проценты, если не None
    similarity_percent = max(0, round(similarity * 100, 1)) if similarity is not None else None

    results.append({
        'ID': row['ID'],
        'сходство_%': similarity_percent,
        'год_мед': table_year,
        'триггер': row['триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 5. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ СРЕДНЕГО ГОДА ПОСТРОЙКИ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 6. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['год_мед'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")



ВВЕДИТЕ СРЕДНИЙ ГОД ЗАСТРОЙКИ В РАДИУСЕ 500 МЕТРОВ
>>> 


ТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ СРЕДНЕГО ГОДА ПОСТРОЙКИ:
--------------------------------------------------------------------------------
 ID сходство_%  год_мед                                                                                                                                         триггер
  1       None   2013.0                                                                                            Подготовка строительной площадки в лесу на Сиреневой
  2       None   1961.0                             Проект ЖК «Сосновый бор» компании «СД Альфа Капитал» / Снос клуба «Отдых» и Дом спорта компанией «СД Альфа Капитал»
  3       None   1987.0 Обновленный Генплан Новосибирска от 24 марта, в котором в зону многоэтажной застройки вошел земельный участок бывшего пионерлагеря «Юный медик»
  4       None   1962.0                                                                                       Снос несколь

коэффициент застройки окружения

In [ ]:
# Код для сравнения по столбцу "застройка_коэф" нормализованная разница

# 1. Пользователь вводит коэффициент застройки
print("\nВВЕДИТЕ КОЭФФИЦИЕНТ ЗАСТРОЙКИ ТЕРРИТОРИИ В РАДИУСЕ 500 МЕТРОВ")
user_input = input(">>> ")

# Проверяем, что введено число
user_coef = None
if user_input:
    try:
        user_coef = float(user_input)
    except ValueError:
        print("Ошибка: введите число от 0 до 1, разделитель точка")
    else:
        if not (0 <= user_coef <= 1):
            print("Ошибка: введите число от 0 до 1, разделитель точка")
            user_coef = None
        else:
            user_coef = float(user_input)

    print(f"\nКоэффициент застройки: {user_coef}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по коэффициентам застройки
def calculate_coefficient_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None

    # Рассчитываем нормализованную разницу
    # Формуле: 1 - (|PA - PB|) / (Pmax - Pmin)
    similarity = 1 - (abs(user_val - table_val) / (max_val - min_val))

    return similarity

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение коэффициента в таблице
valid_coefs = data['застройка_коэф'].dropna()
min_coef = valid_coefs.min()
max_coef = valid_coefs.max()

for idx, row in data.iterrows():
    # Получаем коэффициент из таблицы
    table_coef = row['застройка_коэф']

    # Рассчитываем схожесть
    if user_coef is None or pd.isna(table_coef):
        similarity = None
    else:
        similarity = calculate_coefficient_similarity(user_coef, table_coef, min_coef, max_coef)

    # Преобразуем в проценты, если не None
    similarity_percent = round(similarity * 100, 1) if similarity is not None else None

    results.append({
        'ID': row['ID'],
        'сходство_%': similarity_percent,
        'застройка_коэф': table_coef,
        'триггер': row['триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ КОЭФФИЦИЕНТА ЗАСТРОЙКИ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['застройка_коэф'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


ВВЕДИТЕ КОЭФФИЦИЕНТ ЗАСТРОЙКИ ТЕРРИТОРИИ В РАДИУСЕ 500 МЕТРОВ
>>> 0.15

Коэффициент застройки: 0.15


ТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ КОЭФФИЦИЕНТА ЗАСТРОЙКИ:
--------------------------------------------------------------------------------
 ID  сходство_%  застройка_коэф                                                                                                         триггер
255       100.0        0.150330                                                                      Строительство двух стадионов в Саду Победы
547        99.9        0.149359                                                                                  Вырубка деревьев в парке 1 мая
426        99.9        0.149491                                                          Против строительства дома на игровой площадке во дворе
272        99.9        0.150740                                                Строительство шестиэтажного дома по адресу пер. Дубовский, 9 «В»
465        99.9        0.150768

коэффициент расположения территории в границах города

In [ ]:
# Код для сравнения по столбцу "центральность" нормализованная разница

# 1. Пользователь вводит значение центральности
print("\nВВЕДИТЕ ОТНОШЕНИЕ РАССТОЯНИЯ ОТ ЦЕНТРА ГОРОДА ДО ТЕРРИТОРИИ К РАССТОЯНИЮ ОТ ЦЕНТРА ГОРОДА ДО ГРАНИЦЫ ГОРОДА ПО ТОМУ ЖЕ НАПРАВЛЕНИЮ")
user_input = input(">>> ")

# Проверяем, что введено число
user_centrality = None
if user_input:
    try:
        user_centrality = float(user_input)
    except ValueError:
        print("Ошибка: введите число до 1, разделитель точка")
    else:
        if not (user_centrality <= 1):
            print("Ошибка: введите число до 1, разделитель точка")
            user_centrality = None
        else:
            user_centrality = float(user_input)

    print(f"\nКоэффициент центральности: {user_centrality}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по центральности
def calculate_centrality_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None

    # Рассчитываем нормализованную разницу
    # Формула: 1 - (|PA - PB|) / (Pmax - Pmin)
    similarity = 1 - (abs(user_val - table_val) / (max_val - min_val))

    return similarity

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение центральности в таблице
valid_centralities = data['центральность'].dropna()
min_centrality = valid_centralities.min()
max_centrality = valid_centralities.max()

for idx, row in data.iterrows():
    # Получаем центральность из таблицы
    table_centrality = row['центральность']

    # Рассчитываем схожесть
    if user_centrality is None or pd.isna(table_centrality):
        similarity = None
    else:
        similarity = calculate_centrality_similarity(user_centrality, table_centrality, min_centrality, max_centrality)

    # Преобразуем в проценты, если не None
    similarity_percent = round(similarity * 100, 1) if similarity is not None else None

    results.append({
        'ID': row['ID'],
        'сходство_%': similarity_percent,
        'центральность': table_centrality,
        'триггер': row['триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ЦЕНТРАЛЬНОСТИ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['центральность'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


ВВЕДИТЕ ОТНОШЕНИЕ РАССТОЯНИЯ ОТ ЦЕНТРА ГОРОДА ДО ТЕРРИТОРИИ К РАССТОЯНИЮ ОТ ЦЕНТРА ГОРОДА ДО ГРАНИЦЫ ГОРОДА ПО ТОМУ ЖЕ НАПРАВЛЕНИЮ
>>> 


ТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ЦЕНТРАЛЬНОСТИ:
--------------------------------------------------------------------------------
 ID сходство_%  центральность                                                                                                                                         триггер
  1       None       0.285657                                                                                            Подготовка строительной площадки в лесу на Сиреневой
  2       None       0.495273                             Проект ЖК «Сосновый бор» компании «СД Альфа Капитал» / Снос клуба «Отдых» и Дом спорта компанией «СД Альфа Капитал»
  3       None       0.101672 Обновленный Генплан Новосибирска от 24 марта, в котором в зону многоэтажной застройки вошел земельный участок бывшего пионерлагеря «Юный медик»
  4       None       0.53138

численность населения города

In [ ]:
# Код для сравнения по столбцу "население" логарифмическая нормализованная разница

# 1. Пользователь вводит численность населения
print("\nВВЕДИТЕ ЧИСЛЕННОСТЬ НАСЕЛЕНИЯ")
user_input = input(">>> ")

# Проверяем, что введено число
user_population = None
if user_input:
    try:
        user_population = int(user_input)
    except ValueError:
        print("Ошибка: введите целое число")

    print(f"\nЧисленность населения: {user_population}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по численности населения
def calculate_population_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None

    # Преобразуем значения к целым числам
    user_val = int(user_val)
    table_val = int(table_val)
    min_val = int(min_val)
    max_val = int(max_val)

    # Проверяем, чтобы значения были положительными
    if user_val <= 0 or table_val <= 0 or min_val <= 0 or max_val <= 0:
        return None

    # Рассчитываем логарифмическую нормализованную разницу
    log_user = np.log(user_val)
    log_table = np.log(table_val)
    log_min = np.log(min_val)
    log_max = np.log(max_val)

    # Формула: 1 - (|log(PA) - log(PB)|) / (log(Pmax) - log(Pmin))
    similarity = 1 - (abs(log_user - log_table) / (log_max - log_min))

    return similarity

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение населения в таблице
valid_populations = data['население'].dropna()
min_population = valid_populations.min()
max_population = valid_populations.max()

for idx, row in data.iterrows():
    # Получаем население из таблицы
    table_population = row['население']

    # Рассчитываем схожесть
    if user_population is None or pd.isna(table_population):
        similarity = None
    else:
        similarity = calculate_population_similarity(user_population, table_population, min_population, max_population)

    # Преобразуем в проценты, если не None
    similarity_percent = max(0, round(similarity * 100, 1)) if similarity is not None else None

    results.append({
        'ID': row['ID'],
        'сходство_%': similarity_percent,
        'население': table_population,
        'триггер': row['триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ЧИСЛЕННОСТИ НАСЕЛЕНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['население'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


ВВЕДИТЕ ЧИСЛЕННОСТЬ НАСЕЛЕНИЯ
>>> 5601911

Численность населения: 5601911


ТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ЧИСЛЕННОСТИ НАСЕЛЕНИЯ:
--------------------------------------------------------------------------------
 ID  сходство_%  население                                                                                            триггер
449       100.0    5601911                                               Против строительствамхрама на берегу Матисова канала
452       100.0    5601911                                                                               Против сноса гаражей
451       100.0    5601911                                                   Против высотки "ЛенСпецСМУ" на месте автостоянки
450       100.0    5601911                                     Против строительства многоэтажного дома на месте детского сада
448       100.0    5601911                   Против уплотнительной застройки (жилого комплекса на земле министерства обороны)
364       100.0    560

историческая застройка окружения

In [ ]:
# Код для сравнения по столбцу "истор_здания" бинарное сравнение

# 1. Пользователь вводит наличие исторических зданий
print("\nЕСТЬ ЛИ ИСТОРИЧЕСКИЕ ЗДАНИЯ В ОКРУЖЕНИИ 500 МЕТРОВ?")
print("Введите '1'-есть или '0'-нет")
user_input = input(">>> ")

# Проверяем ввод пользователя
user_hist_building = None
if user_input:
    if user_input.strip() == '1':
        user_hist_building = 1
        print(f"\nПрисутствие исторических зданий: есть")
    elif user_input.strip() == '0':
        user_hist_building = 0
        print(f"\nПрисутствие исторических зданий: нет")
    else:
        print("Ошибка: введите '1' или '0'")
else:
    user_input = float(user_input)

print("\n" + "="*80)

# 2. Функция для расчета сходства по наличию исторических зданий
def calculate_historical_similarity(user_val, table_val):
    """Рассчитывает сходство по бинарному признаку"""

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None

    # Точное совпадение для бинарных значений
    if user_val == table_val:
        return 1.0
    else:
        return 0.0

# 3. Рассчитываем схожесть для каждой строки
results = []

for idx, row in data.iterrows():
    # Получаем значение из таблицы
    table_hist = row['истор_здания']

    # Рассчитываем схожесть
    if user_hist_building is None or pd.isna(table_hist):
        similarity = None
    else:
        similarity = calculate_historical_similarity(user_hist_building, table_hist)

    # Преобразуем в проценты, если не None
    similarity_percent = round(similarity * 100, 1) if similarity is not None else None

    # Преобразуем числовые значения в текст для отображения
    hist_display = "есть" if table_hist == 1 else "нет" if table_hist == 0 else None

    results.append({
        'ID': row['ID'],
        'сходство_%': similarity_percent,
        'истор_здания': hist_display,
        'триггер': row['триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ПО НАЛИЧИЮ ИСТОРИЧЕСКИХ ЗДАНИЙ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['истор_здания'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


ЕСТЬ ЛИ ИСТОРИЧЕСКИЕ ЗДАНИЯ В ОКРУЖЕНИИ 500 МЕТРОВ?
Введите '1'-есть или '0'-нет
>>> ,
Ошибка: введите '1' или '0'


ТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ПО НАЛИЧИЮ ИСТОРИЧЕСКИХ ЗДАНИЙ:
--------------------------------------------------------------------------------
 ID сходство_% истор_здания                                                                                                                                         триггер
  1       None         None                                                                                            Подготовка строительной площадки в лесу на Сиреневой
  2       None         None                             Проект ЖК «Сосновый бор» компании «СД Альфа Капитал» / Снос клуба «Отдых» и Дом спорта компанией «СД Альфа Капитал»
  3       None         None Обновленный Генплан Новосибирска от 24 марта, в котором в зону многоэтажной застройки вошел земельный участок бывшего пионерлагеря «Юный медик»
  4       None         есть             

изменение существующей застройки

In [ ]:
# Код для сравнения по столбцу "год_пост" логарифмическая нормализованная разница

# 1. Пользователь вводит год строительства здания
print("\nВВЕДИТЕ ГОД СТРОИТЕЛЬСТВА СУЩЕСТВУЮЩЕГО НА ТЕРРИТОРИИ ЗДАНИЯ")
print("(если застройка не изменяется, оставьте поле пустым)")
user_input = input(">>> ")

# Проверяем, что введено число
user_build_year = None
if user_input:
    try:
        user_build_year = int(user_input)
    except ValueError:
        print("Ошибка: введите целое число")

    print(f"\nГод строительства здания: {user_build_year}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по годам строительства
def calculate_build_year_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None

    # Преобразуем значения к целым числам
    user_val = int(user_val)
    table_val = int(table_val)
    min_val = int(min_val)
    max_val = int(max_val)

    # Проверяем, чтобы значения были положительными
    if user_val <= 0 or table_val <= 0 or min_val <= 0 or max_val <= 0:
        return None

    # Рассчитываем логарифмическую нормализованную разницу
    log_user = np.log(user_val)
    log_table = np.log(table_val)
    log_min = np.log(min_val)
    log_max = np.log(max_val)

    # Формула: 1 - (|log(PA) - log(PB)|) / (log(Pmax) - log(Pmin))
    similarity = 1 - (abs(log_user - log_table) / (log_max - log_min))

    return similarity

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение года в таблице
valid_years = data['год_пост'].dropna()
min_year = valid_years.min()
max_year = valid_years.max()

for idx, row in data.iterrows():
    # Получаем год из таблицы
    table_year = row['год_пост']

    # Рассчитываем схожесть
    if user_build_year is None or pd.isna(table_year):
        similarity = None
    else:
        similarity = calculate_build_year_similarity(user_build_year, table_year, min_year, max_year)

    # Преобразуем в проценты, если не None
    similarity_percent = max(0, round(similarity * 100, 1)) if similarity is not None else None

    results.append({
        'ID': row['ID'],
        'сходство_%': similarity_percent,
        'год_пост': table_year,
        'триггер': row['триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ГОДА СТРОИТЕЛЬСТВА ЗДАНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('сходство_%', ascending=False).head(10)
print(top_10.to_string(index=False))

# Обрезаем длинные текстовые поля до 50 символов
def truncate_text(value, max_len=50):
        if isinstance(value, str) and len(value) > max_len:
            return value[:max_len] + '...'
        return value

# 5. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['год_пост'] = dict(zip(results_df['ID'], results_df['сходство_%'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к финальному объединению результатов.")


ВВЕДИТЕ ГОД СТРОИТЕЛЬСТВА СУЩЕСТВУЮЩЕГО НА ТЕРРИТОРИИ ЗДАНИЯ
(если застройка не изменяется, оставьте поле пустым)
>>> ,
Ошибка: введите целое число

Год строительства здания: None


ТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ГОДА СТРОИТЕЛЬСТВА ЗДАНИЯ:
--------------------------------------------------------------------------------
 ID сходство_%  год_пост                                                                                                                                         триггер
  1       None       NaN                                                                                            Подготовка строительной площадки в лесу на Сиреневой
  2       None       NaN                             Проект ЖК «Сосновый бор» компании «СД Альфа Капитал» / Снос клуба «Отдых» и Дом спорта компанией «СД Альфа Капитал»
  3       None       NaN Обновленный Генплан Новосибирска от 24 марта, в котором в зону многоэтажной застройки вошел земельный участок бывшего пионерлагеря «Юный

пространственно-функциональное подобные территории

In [ ]:
# подобие территорий композитное взвешенное сходство

# Словарь для преобразования коротких названий в читаемые
criteria_mapping = {
    'сущ_назначение': 'Существующее функциональное использование территории',
    'план_назначение': 'Проектируемое функциональное использование территории',
    'функции_окруж': 'Функциональное использование прилегающих территорий',
    'год_мед': 'Среднее значение годов застройки окружения',
    'застройка_коэф': 'Коэффициент застройки окружения',
    'центральность': 'Коэффициент расположения территории в границах города',
    'население': 'Численность населения города',
    'истор_здания': 'Историческая застройка окружения',
    'год_пост': 'Изменение существующей застройки'
}

criteria_list = ['сущ_назначение', 'план_назначение', 'функции_окруж',
                 'год_мед', 'застройка_коэф', 'центральность',
                 'население', 'истор_здания', 'год_пост']

# 1. Веса по умолчанию (предложенные экспертом)
default_weights = {
    'сущ_назначение': 0.15,
    'план_назначение': 0.15,
    'функции_окруж': 0.10,
    'год_мед': 0.12,
    'застройка_коэф': 0.17,
    'центральность': 0.17,
    'население': 0.04,
    'истор_здания': 0.08,
    'год_пост': 0.02
}

# Проверяем, что сумма весов равна 1
total_weight = sum(default_weights.values())
if abs(total_weight - 1.0) > 0.001:
    # Нормализуем веса, если сумма не равна 1
    default_weights = {k: v/total_weight for k, v in default_weights.items()}

print("ВЕСА КРИТЕРИЕВ ПО УМОЛЧАНИЮ (экспертные оценки):")
print("-" * 80)
for criterion in criteria_list:
    weight = default_weights[criterion]
    full_name = criteria_mapping.get(criterion, criterion)
    print(f"{full_name}: {weight:.3f}")

print(f"\nСумма весов: {sum(default_weights.values()):.3f}")
print("\n" + "="*80)

# 2. Запрос на изменение весов с проверкой ввода
print("\nХотите изменить веса критериев?")
print("Введите '1'-да или '0'-нет")
change_input = input(">>> ").strip()

change_weights = None
if change_input:
    if change_input == '1':
        change_weights = 1
        print(f"\nИзменение весов: включено")
    elif change_input == '0':
        change_weights = 0
        print(f"\nИзменение весов: отключено")
    else:
        print("Ошибка: введите '1' или '0'. Используются веса по умолчанию.")
        change_weights = 0
else:
    print("Используются веса по умолчанию.")
    change_weights = 0

print("\n" + "="*80)

# 3. Настройка пользовательских весов
if change_weights:
    print("\nВВЕДИТЕ ВЕСА ДЛЯ КАЖДОГО КРИТЕРИЯ (от 0 до 1)")
    print("Сумма всех весов должна быть равна 1")
    print("-" * 80)

    custom_weights = {}
    for criterion in criteria_list:
        full_name = criteria_mapping.get(criterion, criterion)

        while True:
            weight_input = input(f"Вес для '{full_name}': ").strip()
            # Проверка на пустой ввод
            if not weight_input:
                print("  Ошибка: введите число от 0 до 1")
                continue
            try:
                weight = float(weight_input)
                # Проверка диапазона
                if weight < 0 or weight > 1:
                    print("  Ошибка: введите число от 0 до 1")
                    continue

                custom_weights[criterion] = weight
                print(f"  ✓ Установлен вес: {weight:.3f}")
                break

            except ValueError:
                print("  Ошибка: введите число от 0 до 1")

    print("\n" + "-" * 80)
    print("ВВЕДЕННЫЕ ВЕСА:")
    print("-" * 80)

    # Суммируем введенные веса
    total_weight = sum(custom_weights.values())
    print(f"Сумма введенных весов: {total_weight:.3f}")

    if abs(total_weight - 1.0) > 0.001:
        # Нормализуем веса
        custom_weights = {k: v/total_weight for k, v in custom_weights.items()}

    print("\nНОРМАЛИЗОВАННЫЕ ВЕСА:")
    print("-" * 80)
    for criterion in criteria_list:
        weight = custom_weights[criterion]
        full_name = criteria_mapping.get(criterion, criterion)
        print(f"{full_name}: {weight:.3f}")

    print(f"\nСумма весов после нормализации: {sum(custom_weights.values()):.3f}")

    weights = custom_weights

else:
    print("\nИспользуются веса по умолчанию")
    weights = default_weights

print("\n" + "="*80)
print("\nРАСЧЕТ СХОДСТВА")
print("-" * 80)

# 4. Функция для расчета итогового сходства
def calculate_total_similarity(row_id, weights_dict):
    """Рассчитывает итоговое взвешенное сходство для конкретного ID"""
    total_similarity = 0
    total_weight = 0
    details = {}

    for criterion, weight in weights_dict.items():
        if criterion in similarity_scores and row_id in similarity_scores[criterion]:
            similarity = similarity_scores[criterion][row_id]
            if similarity is not None:
                try:
                    # Преобразуем в число и проверяем на NaN
                    similarity_float = float(similarity)
                    if not np.isnan(similarity_float):
                        total_similarity += similarity_float * weight
                        total_weight += weight
                        details[criterion] = similarity_float
                    else:
                        details[criterion] = None
                except (ValueError, TypeError):
                    details[criterion] = None
            else:
                details[criterion] = None
        else:
            details[criterion] = None

    # Если нет ни одного критерия с данными
    if total_weight == 0:
        return None, details

    final_similarity = total_similarity / total_weight
    return final_similarity, details

# 5. Рассчитываем итоговое сходство для всех строк
final_results = []

for idx, row in data.iterrows():
    row_id = row['ID']
    final_similarity, details = calculate_total_similarity(row_id, weights)

    # Подсчитываем количество критериев с данными
    criteria_with_data = sum(1 for val in details.values() if val is not None)

    final_results.append({
        'ID': row_id,
        'триггер': row['триггер'],
        'Итоговое_сходство_%': round(final_similarity * 100, 1) if final_similarity is not None else None,
        'Критериев_с_данными': criteria_with_data,
        **{f'{criterion}_%': round(details[criterion] * 100, 1) if details[criterion] is not None else None
           for criterion in criteria_list}
    })

# Создаем DataFrame с результатами
final_df = pd.DataFrame(final_results)

# 6. Сортируем по убыванию итогового сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ ПО ОБЩЕМУ СХОДСТВУ:")
print("(сортировка по убыванию итогового сходства)")
print("-" * 100)

# Создаем список отображаемых колонок, сортируем и форматируем
display_columns = ['ID', 'триггер', 'Итоговое_сходство_%', 'Критериев_с_данными'] + [f'{criterion}_%' for criterion in criteria_list]
top_10_final = final_df.sort_values('Итоговое_сходство_%', ascending=False).head(10)

def format_cell(value, col_name):
    if col_name.endswith('_%') and value is not None and isinstance(value, (int, float)):
        return f"{value}%"
    elif value is None:
        return "-"
    else:
        return str(value)

# Форматируем каждую ячейку отдельно
formatted_rows = []
for _, row in top_10_final[display_columns].iterrows():
    formatted_row = []
    for col in display_columns:
        value = row[col]
        formatted_row.append(format_cell(value, col))
    formatted_rows.append(formatted_row)

# Выводим таблицу с выравниванием
from tabulate import tabulate
print(tabulate(formatted_rows, headers=display_columns, tablefmt='grid'))

# 7. Выводим детализацию для лучшего результата
print("\n" + "="*80)
print("ДЕТАЛИЗАЦИЯ ЛУЧШЕГО РЕЗУЛЬТАТА:")
print("-" * 80)

best_result = top_10_final.iloc[0]
print(f"ID: {best_result['ID']}")
print(f"Триггер конфликта: {best_result['триггер']}")
print(f"Итоговое сходство: {best_result['Итоговое_сходство_%']}%")
print(f"Учтено критериев: {best_result['Критериев_с_данными']}/{len(criteria_list)}")
print("\nСходство по критериям:")
for criterion in criteria_list:
    similarity = best_result[f'{criterion}_%']
    weight = weights[criterion]
    if similarity is not None:
        print(f"  {criterion}: {similarity}% (вес: {weight:.3f})")

print("\n" + "="*80)
print("РАСЧЕТ ЗАВЕРШЕН. НАЙДЕНО 10 НАИБОЛЕЕ ПОХОЖИХ ТЕРРИТОРИЙ.")

# 8. Сохраняем словарь с весами для возможного использования
if 'criteria_weights' not in globals():
    criteria_weights = {}
criteria_weights = weights.copy()


ВЕСА КРИТЕРИЕВ ПО УМОЛЧАНИЮ (экспертные оценки):
--------------------------------------------------------------------------------
Существующее функциональное использование территории: 0.150
Проектируемое функциональное использование территории: 0.150
Функциональное использование прилегающих территорий: 0.100
Среднее значение годов застройки окружения: 0.120
Коэффициент застройки окружения: 0.170
Коэффициент расположения территории в границах города: 0.170
Численность населения города: 0.040
Историческая застройка окружения: 0.080
Изменение существующей застройки: 0.020

Сумма весов: 1.000


Хотите изменить веса критериев?
Введите '1'-да или '0'-нет
>>> 1

Изменение весов: включено


ВВЕДИТЕ ВЕСА ДЛЯ КАЖДОГО КРИТЕРИЯ (от 0 до 1)
Сумма всех весов должна быть равна 1
--------------------------------------------------------------------------------
Вес для 'Существующее функциональное использование территории': 0.2
  ✓ Установлен вес: 0.200
Вес для 'Проектируемое функциональное использовани

NameError: name 'data' is not defined

сопоставление конфликтологической экспертизы с пространственно-функциональными характеристиками территории

In [ ]:
# Загрузка второго файла с дополнительной информацией о конфликтах
print("ЗАГРУЗКА ДОПОЛНИТЕЛЬНОЙ ИНФОРМАЦИИ О КОНФЛИКТАХ")
print("\n"+"="*80)

try:
    # Загружаем файл all_con
    df_con = pd.read_excel('all_con.xlsx')
    print(f"\nКоличество строк: {df_con.shape[0]}, столбцов: {df_con.shape[1]}")

    # Проверяем наличие столбца ID
    if 'ID' in df_con.columns:
        print(f"Столбец ID найден, уникальных ID: {df_con['ID'].nunique()}")
    else:
        print("ВНИМАНИЕ: Столбец ID не найден в файле all_con.xlsx")
        print("Доступные столбцы:")
        for i, col in enumerate(df_con.columns, 1):
            print(f"{i}. {col}")

        # Просим указать столбец с ID
        id_col = input("\nВведите название столбца, содержащего ID: ").strip()
        if id_col in df_con.columns:
            df_con = df_con.rename(columns={id_col: 'ID'})
            print(f"Столбец '{id_col}' переименован в 'ID'")
        else:
            raise ValueError(f"Столбец '{id_col}' не найден в файле")

except Exception as e:
    print("Ошибка при загрузке файла all_con.xlsx. Убедитесь, что файл загружен в Google Colab")

print("\n" + "="*80)


# 1. Выводим информацию по 3 самым похожим конфликтам
print("\nПОДРОБНАЯ ИНФОРМАЦИЯ ПО 3 САМЫМ ПОХОЖИМ КОНФЛИКТАМ")

for i, (_, row) in enumerate(top_10_final.head(3).iterrows(), 1):
        conflict_id = row['ID']
        similarity = row['Итоговое_сходство_%']
        trigger = row['триггер']

        print(f"\n{'='*80}")
        print(f"Конфликт #{i} | ID: {conflict_id}")
        print(f"Сходство с проектом: {similarity}%")
        print(f"{'-'*80}")

        # Находим соответствующую запись в втором файле
        conflict_details = df_con[df_con['ID'] == conflict_id]

        if not conflict_details.empty:
            # Выводим все доступные поля (кроме технических)
            conflict_row = conflict_details.iloc[0]

            # Выбираем столбцы для подробного вывода
            detail_cols = [col for col in df_con.columns
                          if col != 'ID' and
                          not pd.isna(conflict_row[col]) and
                          str(conflict_row[col]).strip() != '']

            for col in detail_cols:
                value = conflict_row[col]
                if isinstance(value, float) and value.is_integer():
                    value = int(value)

                # Форматируем длинные тексты
                if isinstance(value, str) and len(str(value)) > 100:
                    # Разбиваем длинный текст на строки
                    print(f"\n{col}:")
                    words = str(value).split()
                    line = ""
                    for word in words:
                        if len(line) + len(word) + 1 > 100:
                            print(f"{line}")
                            line = word
                        else:
                            line += " " + word if line else word
                    if line:
                        print(f"{line}")
                else:
                    print(f"\n{col}: {value}")
        else:
            print(f"\nДополнительная информация для конфликта ID {conflict_id} не найдена.")

        # Также выводим сходство по критериям
        print(f"\nСходство по критериям:")
        for criterion in criteria_list:
            col_name = f'{criterion}_%'
            if col_name in row:
                similarity_value = row[col_name]
                if similarity_value is not None:
                    full_name = criteria_mapping.get(criterion, criterion)
                    print(f"  - {full_name}: {similarity_value}%")

print("\n" + "="*80)

# 2. Сопоставление топ-10 с дополнительной информацией
if not df_con.empty and 'ID' in df_con.columns:
    # Список нужных столбцов
    required_columns = [
        'ID',
        'Описание конфликта',
        'Триггеры',
        'Акторы',
        'Организаторы',
        'Интересанты проекта',
        'Форма протеста',
        'Исход конфликта',
        'Масштаб конфликта',
        'Город',
        'Адрес',
        'Год начала конфликта',
        'Год окончания конфликта'
    ]

    # Проверяем, какие столбцы есть в файле
    available_cols = []
    missing_cols = []

    for col in required_columns:
        if col in df_con.columns:
            available_cols.append(col)
        elif col != 'ID':
            missing_cols.append(col)


    # Объединяем данные
    merge_cols = ['ID'] + [col for col in available_cols if col != 'ID']

    top_10_merged = pd.merge(
        top_10_final[['ID', 'триггер', 'Итоговое_сходство_%']],
        df_con[merge_cols],
        on='ID',
        how='left'
    )

    # Переименовываем столбец сходства
    top_10_merged = top_10_merged.rename(columns={'Итоговое_сходство_%': 'Сходство с проектом'})

    # Обрезаем длинные текстовые поля до 50 символов
    def truncate_text(value, max_len=50):
        if isinstance(value, str) and len(value) > max_len:
            return value[:max_len] + '...'
        return value

    # Применяем обрезку ко всем текстовым столбцам
    for col in top_10_merged.columns:
        if col not in ['ID', 'Сходство', 'Год начала конфликта', 'Год окончания конфликта']:
            top_10_merged[col] = top_10_merged[col].apply(truncate_text)

    # Устанавливаем порядок столбцов
    display_order = ['ID', 'Сходство', 'Триггеры', 'Акторы',
                    'Интересанты проекта', 'Форма протеста',
                    'Исход конфликта', 'Город', 'Адрес']

    # Оставляем только существующие столбцы
    display_order = [col for col in display_order if col in top_10_merged.columns]

    print("\nДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ ТОП-10:")
    print("\n" + "="*80)
    print(top_10_merged[display_order].to_string(index=False))

    print("\n" + "="*80)

else:
    print("✗ Не удалось загрузить дополнительную информацию")

print("\nАНАЛИЗ ЗАВЕРШЕН")
print("\n" + "="*80)


ЗАГРУЗКА ДОПОЛНИТЕЛЬНОЙ ИНФОРМАЦИИ О КОНФЛИКТАХ


Количество строк: 614, столбцов: 17
Столбец ID найден, уникальных ID: 614


ПОДРОБНАЯ ИНФОРМАЦИЯ ПО 3 САМЫМ ПОХОЖИМ КОНФЛИКТАМ

Конфликт #1 | ID: 198
Сходство с проектом: 99.4%
--------------------------------------------------------------------------------

Описание конфликта: за закрытие азербайджанского ресторана "Тебриз

Триггеры: Работа азербайджанского ресторана "Тебриз"

Организаторы: местные жители

Интересанты проекта: местная власть, 

Форма протеста: пикет

Исход конфликта: Проект есть  / ресторан работает

Масштаб конфликта: 20

Субъект: Санкт-Петербург 

Город: Санкт-Петербург 

Численность населения: 5601911

Адрес: Санкт-Петербург, ул. Седова,154

Координаты широта: 59.864784

Координаты долгота: 30.445583

Год начала конфлитка: 2013

Год окончания  конфлитка: 2013

Сходство по критериям:
  - Существующее функциональное использование территории: 100.0%
  - Проектируемое функциональное использование территории: 10.0%
  - Ф

скачивание полной сводной таблицы лучших совпадений

In [ ]:
# Создание финального CSV файла с детальной информацией

print("СОЗДАНИЕ ФИНАЛЬНОГО CSV ФАЙЛА")
print("="*80)

# 1.  Режим сохранения
print("\nВЫБЕРИТЕ РЕЖИМ СОХРАНЕНИЯ:")
print("0 - Сохранить только топ-10 лучших совпадений")
print("1 - Сохранить ВСЕ рассчитанные конфликты")
save_mode = input("Введите 0 или 1: ").strip()

# Определяем, какой датафрейм использовать как источник
if save_mode == '1':
    # Все данные (отсортированные по убыванию сходства)
    source_df = final_df.sort_values('Итоговое_сходство_%', ascending=False).copy()
    mode_name = "all"
    print("\n✓ Выбран режим: СОХРАНИТЬ ВСЕ ДАННЫЕ")
else:
    # По умолчанию (или если введено 1) — топ-10
    source_df = top_10_final.copy()
    mode_name = "top10"
    print("\n✓ Выбран режим: СОХРАНИТЬ ТОП-10")

# 1. Проверяем наличие всех необходимых столбцов
all_con_columns = [
    'Описание конфликта', 'Акторы', 'Организаторы', 'Интересанты проекта',
    'Форма протеста', 'Исход конфликта', 'Масштаб конфликта', 'Субъект',
    'Город', 'Численность населения', 'Адрес', 'Координаты широта',
    'Координаты долгота', 'Год начала конфликта', 'Год окончания конфликта'
]

print("Проверка наличия столбцов в файле all_con:")
available_cols = []
missing_cols = []

for col in all_con_columns:
    if col in df_con.columns:
        available_cols.append(col)
        print(f"✓ {col}")
    else:
        missing_cols.append(col)
        print(f"✗ {col}")

if missing_cols:
    print(f"\nВНИМАНИЕ: Отсутствуют {len(missing_cols)} столбцов:")
    for col in missing_cols:
        print(f"  - {col}")

print("\n" + "="*80)

# 2. Создаем финальный DataFrame
print("\nФормирование финального DataFrame...")

# Берем конфликты
selected_ids = source_df['ID'].tolist()

# Фильтруем данные по ID
selected_par = data[data['ID'].isin(selected_ids)].copy()
selected_con = df_con[df_con['ID'].isin(selected_ids)].copy()

# Добавляем финальное сходство
selected_par = pd.merge(selected_par,
                        source_df[['ID', 'Итоговое_сходство_%', 'триггер']],
                        on='ID',
                        how='left')

# 3. Создаем список столбцов в нужном порядке
final_columns = []

# Добавляем базовые столбцы
final_columns.append('ID')
final_columns.append('триггер')
final_columns.append('Итоговое_сходство_%')

# Добавляем столбцы из all_con
for col in all_con_columns:
    if col in available_cols:
        final_columns.append(col)

# 4. Создаем словарь для соответствия названий критериев
criteria_display_names = {
    'сущ_назначение': 'Существующее функциональное использование территории',
    'план_назначение': 'Проектируемое функциональное использование территории',
    'функции_окруж': 'Функциональное использование прилегающих территорий',
    'год_мед': 'Среднее значение годов застройки окружения',
    'застройка_коэф': 'Коэффициент застройки окружения',
    'центральность': 'Коэффициент расположения территории в границах города',
    'население': 'Численность населения города',
    'истор_здания': 'Историческая застройка окружения',
    'год_пост': 'Изменение существующей застройки'
}

# 5. Объединяем данные
final_df_export = pd.DataFrame()

original_values_dict = {}
for _, data_row in selected_par.iterrows():
    conflict_id = data_row['ID']
    values = {}
    for criterion in criteria_list:
        if criterion in data_row:
            values[criterion] = data_row[criterion]   # берём исходное значение как есть
        else:
            values[criterion] = None
    original_values_dict[conflict_id] = values

for idx, row in source_df.iterrows():
    conflict_id = row['ID']

    # Создаем словарь для этой строки
    row_data = {}

    # Добавляем базовые данные
    row_data['ID'] = conflict_id
    row_data['триггер'] = row['триггер']
    row_data['Итоговое_сходство_%'] = row['Итоговое_сходство_%']

    # Добавляем данные из all_con
    con_row = df_con[df_con['ID'] == conflict_id]
    if not con_row.empty:
        con_data = con_row.iloc[0]
        for col in available_cols:
            if col in con_data:
                row_data[col] = con_data[col]

    # Добавляем критерии и их проценты поочередно
    orig_vals = original_values_dict.get(conflict_id, {})
    for criterion in criteria_list:
        # Полное название критерия
        row_data[f'{criterion}_value'] = orig_vals.get(criterion, None)

        # Процент сходства
        criterion_pct_col = f'{criterion}_%'
        if criterion_pct_col in row:
            row_data[f'{criterion}_%'] = row[criterion_pct_col]
        else:
            row_data[f'{criterion}_%'] = None

    # Добавляем строку в финальный DataFrame
    final_df_export = pd.concat([final_df_export, pd.DataFrame([row_data])], ignore_index=True)

# 6. Переименовываем столбцы для лучшей читаемости
rename_dict = {
    'Итоговое_сходство_%': 'Финальный процент соответствия',
    'сущ_назначение_value': 'Критерий: Существующее функциональное использование территории',
    'сущ_назначение_%': 'Сходство: Существующее функциональное использование территории',
    'план_назначение_value': 'Критерий: Проектируемое функциональное использование территории',
    'план_назначение_%': 'Сходство: Проектируемое функциональное использование территории',
    'функции_окруж_value': 'Критерий: Функциональное использование прилегающих территорий',
    'функции_окруж_%': 'Сходство: Функциональное использование прилегающих территорий',
    'год_мед_value': 'Критерий: Среднее значение годов застройки окружения',
    'год_мед_%': 'Сходство: Среднее значение годов застройки окружения',
    'застройка_коэф_value': 'Критерий: Коэффициент застройки окружения',
    'застройка_коэф_%': 'Сходство: Коэффициент застройки окружения',
    'центральность_value': 'Критерий: Коэффициент расположения территории в границах города',
    'центральность_%': 'Сходство: Коэффициент расположения территории в границах города',
    'население_value': 'Критерий: Численность населения города',
    'население_%': 'Сходство: Численность населения города',
    'истор_здания_value': 'Критерий: Историческая застройка окружения',
    'истор_здания_%': 'Сходство: Историческая застройка окружения',
    'год_пост_value': 'Критерий: Изменение существующей застройки',
    'год_пост_%': 'Сходство: Изменение существующей застройки'
}

final_df_export = final_df_export.rename(columns=rename_dict)

# 7. Сохраняем в CSV
from datetime import datetime
import os

if mode_name == "top10":
    base_filename = 'top10_conflicts'
else:
    base_filename = 'all_conflicts'

# Получаем ID первого конфликта из топ-10
if not source_df.empty:
    first_id = source_df.iloc[0]['ID']
    print(f"ID первого конфликта: {first_id}")

    # Создаем имя файла с ID первого конфликта
    output_filename = f"{base_filename}_{first_id}.csv"
    final_df_export.to_csv(output_filename, index=False, encoding='utf-8-sig')


print(f"\n✓ Файл успешно создан: {output_filename}")
print(f"✓ Количество строк: {len(final_df_export)}")
print(f"✓ Количество столбцов: {len(final_df_export.columns)}")

print("\nСтруктура файла:")
print("-" * 80)
print("1. ID конфликта")
print("2. Триггер конфликта")
print("3. Финальный процент соответствия")
print("4. Данные из файла all_con (15 столбцов)")
print("5. Чередование: Критерий -> Сходство (9 пар = 18 столбцов)")
print(f"Итого: {len(final_df_export.columns)} столбцов")

print("\n" + "="*80)

# 9. Показываем первые несколько строк для проверки
print("\nПРЕДПРОСМОТР ФАЙЛА (первые 2 строки):")
print("-" * 80)

# Показываем только первые несколько столбцов для наглядности
preview_cols = list(final_df_export.columns)[:8]
print(final_df_export[preview_cols].head(2).to_string(index=False))

print("\n" + "="*80)

СОЗДАНИЕ ФИНАЛЬНОГО CSV ФАЙЛА

ВЫБЕРИТЕ РЕЖИМ СОХРАНЕНИЯ:
0 - Сохранить только топ-10 лучших совпадений
1 - Сохранить ВСЕ рассчитанные конфликты
Введите 0 или 1: 1

✓ Выбран режим: СОХРАНИТЬ ВСЕ ДАННЫЕ
Проверка наличия столбцов в файле all_con:
✓ Описание конфликта
✓ Акторы
✓ Организаторы
✓ Интересанты проекта
✓ Форма протеста
✓ Исход конфликта
✓ Масштаб конфликта
✓ Субъект
✓ Город
✓ Численность населения
✓ Адрес
✓ Координаты широта
✓ Координаты долгота
✗ Год начала конфликта
✗ Год окончания конфликта

ВНИМАНИЕ: Отсутствуют 2 столбцов:
  - Год начала конфликта
  - Год окончания конфликта


Формирование финального DataFrame...
ID первого конфликта: 380

✓ Файл успешно создан: all_conflicts_380.csv
✓ Количество строк: 611
✓ Количество столбцов: 34

Структура файла:
--------------------------------------------------------------------------------
1. ID конфликта
2. Триггер конфликта
3. Финальный процент соответствия
4. Данные из файла all_con (15 столбцов)
5. Чередование: Критерий -> Сходс